In [1]:
import spatialdata as sd
import spatialdata_plot
import matplotlib.pyplot as plt
import numpy as np
import os
from skimage import io
import pandas as pd
import sopa
from skimage import io as skio
from skimage.measure import regionprops_table #library for calculate centroids
from spatialdata.models import Labels2DModel, ShapesModel
import rasterio.features
import shapely.geometry
import geopandas as gpd

/home/stefano/miniconda3/envs/analysis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [132]:
#Import data
#work with a copy since modification on spatial data object are in place write on disk
data_path = "/home/stefano/Documents/Spatial-Transcriptomic/data/blocco1_sham_(Copy)"
sdata = sd.read_zarr(data_path)


no parent found for <ome_zarr.reader.Label object at 0x77c72254da50>: None


In [134]:
#read and upload mask obtained through cellpose
cellpose_mask= skio.imread("/home/stefano/Documents/Spatial-Transcriptomic/images/blocco1_sham_masks.tif")

In [135]:
#obtain the transformation to apply to the mask from full res image
original_coords = sd.transformations.get_transformation(sdata.images['blocco1_full_image'], get_all= True)
original_coords

{'blocco1': Sequence 
     Translation (c, y, x)
         [   0. 8000.    0.]
     Identity }

In [5]:
from shapely.geometry import shape

In [136]:
#extract geometrical boundaries of the segmented fibers from the mask
shapes_generator = rasterio.features.shapes(cellpose_mask.astype(np.int32), mask=(cellpose_mask>0))
poligons = []
ids = []

for geom, value in shapes_generator:
    poligons.append(shape(geom))
    ids.append(int(value))

gdf = gpd.GeoDataFrame({'geometry': poligons, 'fiber_id':ids})


#Convert in a ShapedModel element in order to add to the Spatial Data object
#shapes_element = ShapesModel.parse(gdf, transformations=original_coords)
#sdata.shapes['fiber_polygons'] = shapes_element
#shapes_element

Here we have a problem: there are 2636 segmented fibers but  rasterio.features.shapes defined 2653 elements, we need to merge element with the same label since some fibers have benn fragmented in more geometric pieces.

In [137]:
#SOLUTION OF THE PROBLEM: merge the geometry which have the same fiber label
gdf_cleaned = gdf.dissolve(by='fiber_id', as_index=True)


shapes_element = ShapesModel.parse(gdf_cleaned, transformations=original_coords) #parse coordinate of the polygons (boundary of fibers) respect to the coords of the full res image

sdata.shapes['fiber_polygons'] = shapes_element #add the information of the coordinates of the shape to the spatial data object
sdata['fiber_polygons'].index = sdata['fiber_polygons'].index.astype(str) #set the index as str to see if the fiber id is mantained after aggregation

In [138]:
#aggregation with sopa
sopa.aggregate(sdata,key_added='segmented_fiber' ,bins_key='square_002um', shapes_key='fiber_polygons',expand_radius_ratio=0, min_transcripts=10,
            min_intensity_ratio=0.15, no_overlap=True)

/home/stefano/miniconda3/envs/analysis/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
[INFO] (sopa.aggregation.aggregation) 24 cell(s) not passing filtering due to transcript count < 10
[INFO] (sopa.aggregation.channels) Aggregating channels intensity over 2636 cells with mode='average'


[########################################] | 100% Completed | 57.76 s


[INFO] (sopa.aggregation.aggregation) 0 cell(s) not passing filtering due to mean channel intensity < 30.17


ValueError: Element segmented_fiber is not found in the Zarr store associated with the SpatialData object.

In [ ]:
""""
Here we can notice which the numeber of elements of fiber_polygons on the shapes subcategories reduced from 2636 to 2612, this is because sopa have some filters(e.g. exclude empty fibers,
exlude fibers wich have the number of bins below a certain treshold 
"""

'"\nHere we can notice which the numeber of elements of fiber_polygons on the shapes subcategories reduced from 2636 to 2612, this is because sopa have some filters(e.g. exclude empty fibers,\nexlude fibers wich have the number of bins below a certain treshold \n'

In [114]:
sdata['fiber_polygons']

,geometry
aaaaaaal-1,"POLYGON ((5244 492, 5244 496, 5224 496, 5224 5..."
aaaaaaan-1,"POLYGON ((5556 512, 5556 516, 5544 516, 5544 5..."
aaaaaaap-1,"POLYGON ((5148 520, 5148 524, 5132 524, 5132 5..."
aaaaaabb-1,"POLYGON ((5604 536, 5604 548, 5608 548, 5608 5..."
aaaaaabf-1,"POLYGON ((5024 620, 5024 624, 5020 624, 5020 6..."
...,...
aaaaakeg-1,"POLYGON ((6188 6524, 6188 6536, 6184 6536, 618..."
aaaaakeh-1,"POLYGON ((9892 6552, 9892 6556, 9888 6556, 988..."
aaaaakei-1,"POLYGON ((5864 6556, 5864 6580, 5868 6580, 586..."
aaaaakej-1,"POLYGON ((5692 6580, 5692 6584, 5684 6584, 568..."


In [142]:
gdf_cleaned.index

Index(['aaaaaaaa-1', 'aaaaaaab-1', 'aaaaaaac-1', 'aaaaaaad-1', 'aaaaaaae-1',
       'aaaaaaaf-1', 'aaaaaaag-1', 'aaaaaaah-1', 'aaaaaaai-1', 'aaaaaaaj-1',
       ...
       'aaaaakec-1', 'aaaaaked-1', 'aaaaakee-1', 'aaaaakef-1', 'aaaaakeg-1',
       'aaaaakeh-1', 'aaaaakei-1', 'aaaaakej-1', 'aaaaakek-1', 'aaaaakel-1'],
      dtype='str', length=2636)

In [ ]:
#test if boundary conflict are resolved differently if we set no_overlap=False respect if it setted to True

# RCTD
Create an AnnData object which contain:
* fiber expr matrix
* centroid matrix

In [ ]:
#Obtain centroid matrix, with coordinate centroids of each fiber
polygon_centroids = sdata.shapes['fiber_polygons'].copy()
polygon_centroids['X'] = polygon_centroids.geometry.centroid.x
polygon_centroids['Y'] = polygon_centroids.centroid.y
polygon_centroids = polygon_centroids[['X','Y']] #hold only X and Y values

In [161]:
polygon_centroids

,X,Y
aaaaaaal-1,5322.037799,584.034464
aaaaaaan-1,5556.751761,655.823944
aaaaaaap-1,5182.376238,665.704290
aaaaaabb-1,5662.666667,644.089980
aaaaaabf-1,5132.249639,796.635181
...,...,...
aaaaakeg-1,6294.929341,6579.585629
aaaaakeh-1,9905.866071,6646.848214
aaaaakei-1,5941.203252,6598.008130
aaaaakej-1,5754.668132,6619.934066


In [58]:
fibers_string_id=  sdata.tables['segmented_fiber'].obs.index
fiber_id_df = pd.DataFrame(
    index=pd.Index(fibers_string_id, dtype=object)
)
fiber_id_df.index.name = "fiber_id" 

genes_names_string = [str(g) for g in sdata.tables['segmented_fiber'].var.index]
gene_name_df = pd.DataFrame(
    index = pd.Index(genes_names_string, dtype=object)
)